In [0]:
%run "../00_common/01_environment_config"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Drivers Dimension
# MAGIC
# MAGIC 1. Read Silver `drivers`
# MAGIC 2. Read Gold `ref_nationality_region`
# MAGIC 3. Join using `nationality`
# MAGIC 4. Select required columns
# MAGIC 5. Write to Gold `dim_drivers`


# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

drivers_path = f"{s3_root_path}/Silver/Kalashri/drivers"

ref_path = f"{s3_root_path}/Gold/Kalashri/ref_nationality_region"

target_path = f"{s3_root_path}/Gold/Kalashri/dim_drivers"

print("Drivers Path   :", drivers_path)
print("Reference Path :", ref_path)
print("Target Path    :", target_path)


# COMMAND ----------

# ============================================================
# STEP 1 - READ SILVER DRIVERS
# ============================================================

drivers_df = (
    spark.read
        .format("delta")
        .load(drivers_path)
)

display(drivers_df.limit(20))


# COMMAND ----------

drivers_df.printSchema()


# COMMAND ----------

# ============================================================
# STEP 2 - READ GOLD NATIONALITY-REGION REFERENCE
# ============================================================

ref_nationality_region_df = (
    spark.read
        .format("delta")
        .load(ref_path)
)

display(ref_nationality_region_df.limit(20))


# COMMAND ----------

ref_nationality_region_df.printSchema()


# COMMAND ----------

# ============================================================
# STEP 3 - JOIN DRIVERS WITH NATIONALITY REGION
# ============================================================
# LEFT JOIN keeps all drivers.
# If nationality does not have a matching region,
# nationality_region will be NULL.
# ============================================================

dim_drivers_df = (
    drivers_df
        .join(
            ref_nationality_region_df,
            drivers_df.nationality ==
            ref_nationality_region_df.nationality,
            "left"
        )
        .select(
            drivers_df.driver_id,
            drivers_df.full_name.alias("driver_name"),
            drivers_df.date_of_birth,
            drivers_df.nationality,
            ref_nationality_region_df.region.alias(
                "nationality_region"
            )
        )
)

display(dim_drivers_df.limit(20))


# COMMAND ----------

# ============================================================
# STEP 4 - CHECK RECORD COUNTS
# ============================================================

print("Silver Driver Records :", drivers_df.count())
print("Gold Driver Records   :", dim_drivers_df.count())


# COMMAND ----------

# ============================================================
# STEP 5 - WRITE TO GOLD
# ============================================================

(
    dim_drivers_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(target_path)
)


# COMMAND ----------

# ============================================================
# STEP 6 - VALIDATE GOLD DATA
# ============================================================

gold_dim_drivers_df = (
    spark.read
        .format("delta")
        .load(target_path)
)

display(gold_dim_drivers_df.limit(20))


# COMMAND ----------

gold_dim_drivers_df.printSchema()


# COMMAND ----------

print("Silver Driver Records :", drivers_df.count())
print("Gold Driver Records   :", gold_dim_drivers_df.count())

print("Gold dim_drivers successfully written to:")
print(target_path)

In [0]:
dbutils.fs.rm(
    "s3://bizf1project123/Gold/Kalashri/results",
    True
)

In [0]:
display(
    dbutils.fs.ls("s3://bizf1project123/Gold/Kalashri")
)